In [10]:
# !pip install fastparquet

   ---------------------------------------- 0.0/698.1 kB ? eta -:--:--
   ---------------------------------------- 698.1/698.1 kB 13.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 30.8 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]



In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin

BASE_URL = "https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.03/output/"
session = requests.Session()


def format_size(size):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} PB"


def get_links(url):
    r = session.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    folders = []
    files = []
    for a in soup.find_all("a"):
        href = a.get("href", "")
        if href in ("../", "/"):
            continue
        if href.endswith("/") and not href.startswith("/"):
            folders.append(href.rstrip("/"))
        elif href.endswith(".parquet"):
            files.append(href)

    return folders, files


def get_folder_info(url):
    folders, files = get_links(url)
    total_size = 0
    file_count = 0
    for filename in files:
        file_url = urljoin(url, filename)
        r = session.head(file_url, allow_redirects=True)
        total_size += int(r.headers.get("Content-Length", 0))
        file_count += 1

    for folder in folders:
        count, size = get_folder_info(
            urljoin(url, folder + "/")
        )
        file_count += count
        total_size += size

    return file_count, total_size


def download_folder(url, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    folders, files = get_links(url)
    for filename in files:
        file_url = urljoin(url, filename)
        save_path = save_dir / filename
        if save_path.exists():
            print("이미 있음:", save_path)
            continue
        print("다운로드:", filename)
        with session.get(file_url, stream=True) as r:
            r.raise_for_status()
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
    for folder in folders:
        download_folder(
            urljoin(url, folder + "/"),
            save_dir / folder
        )


# 1. 목록만 빠르게 출력
datasets, _ = get_links(BASE_URL)
print("=== Open Targets 데이터 목록 ===")
for i, name in enumerate(datasets):
    print(f"{i:2}: {name}")

# 2. 원하는 데이터 선택
nums = input("\n다운로드할 번호 (예: 2,5,10): ")

selected = [
    int(x.strip())
    for x in nums.split(",")
]


# 3. 선택한 것만 용량 계산
total = 0
print("\n선택 데이터 용량 확인\n")
for num in selected:
    name = datasets[num]
    count, size = get_folder_info(
        urljoin(BASE_URL, name + "/")
    )
    total += size
    print(
        f"{name:40} "
        f"{count} files | "
        f"{format_size(size)}"
    )

print("\n총 예상 용량:", format_size(total))


# 4. 다운로드
confirm = input("\n다운로드할까요? (y/n): ")
if confirm.lower() == "y":
    for num in selected:
        name = datasets[num]
        download_folder(
            urljoin(BASE_URL, name + "/"),
            Path("opentargets") / name
        )

    print("\n다운로드 완료")

=== Open Targets 데이터 목록 ===
 0: association_by_datasource_direct
 1: association_by_datasource_indirect
 2: association_by_datatype_direct
 3: association_by_datatype_indirect
 4: association_by_overall_indirect
 5: association_overall_direct
 6: biosample
 7: colocalisation_coloc
 8: colocalisation_ecaviar
 9: credible_set
10: disease
11: disease_hpo
12: disease_phenotype
13: drug_indication
14: drug_mechanism_of_action
15: drug_molecule
16: drug_warning
17: evidence
18: expression
19: go
20: interaction
21: interaction_evidence
22: known_drug
23: l2g_prediction
24: literature
25: literature_vector
26: mouse_phenotype
27: openfda_significant_adverse_drug_reactions
28: openfda_significant_adverse_target_reactions
29: pharmacogenomics
30: reactome
31: so
32: study
33: target
34: target_essentiality
35: target_prioritisation
36: variant


In [2]:
# sample
import pandas as pd
from IPython.display import display
df = pd.read_parquet("./opentargets/association_by_datasource_direct/")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: './opentargets/association_by_datasource_direct/'